# ContextMed: Globally Informed, Locally Accurate Clinical AI

**Competition:** MedGemma Impact Challenge  
**Tracks:** Main Track + Agentic Workflow Prize  

---

**ContextMed** is an agentic clinical decision-support system that delivers the **right guideline, right drug, and right source** based on the physician's geography, specialty, and experience level.

A physician in **Berlin** gets ESC/AWMF/EMA guidelines. A physician in **Boston** gets ACC/AHA/FDA guidelines. Same patient, same condition — **different regulatory context, different correct answer**.

## Architecture

| Component | Technology |
|-----------|------------|
| Clinical LLM | MedGemma 4B (4-bit quantized, local GPU) |
| Agentic Framework | LangGraph (Planner → Retriever → Reasoner → Formatter) |
| Literature Search | PubMed E-Utilities (free API) |
| Drug Safety | OpenFDA (free API) |
| Guidelines | Tavily (geography-aware domain filtering) |
| Safety | Real-time allergy cross-reactivity checking |
| Memory | Multi-turn conversation + clinical notepad |
| API | FastAPI with SSE streaming |

## What Makes This Agentic

| Feature | Implementation |
|---------|----------------|
| **LLM-driven planning** | MedGemma classifies intent and selects tools |
| **Parallel tool execution** | All tools run async concurrently |
| **Evidence-grounded reasoning** | Response synthesised from retrieved context |
| **Context-adaptive** | Experience level, specialty, and geography shape output |
| **Memory-aware** | Multi-turn conversations with clinical notepad |

## 1. Install Dependencies

In [ ]:
!pip install -q huggingface_hub transformers accelerate bitsandbytes
!pip install -q langgraph langchain-core
!pip install -q pydantic aiohttp nest_asyncio
!pip install -q tavily-python
!pip install -q fastapi uvicorn hypercorn pyngrok

## 2. Load API Keys & Login

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle/working")

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("huggingface")
    TAVILY_API_KEY = secrets.get_secret("TAVILY_API_KEY")
    NGROK_TOKEN = secrets.get_secret("ngrok")
else:
    HF_TOKEN = os.getenv("HF_TOKEN", "")
    TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "")
    NGROK_TOKEN = os.getenv("NGROK_TOKEN", "")

from huggingface_hub import login
login(token=HF_TOKEN)

print(f"HuggingFace: logged in")
print(f"Tavily API key: {'loaded' if TAVILY_API_KEY else 'missing'}")
print(f"Ngrok token: {'loaded' if NGROK_TOKEN else 'missing'}")

## 3. Check GPU

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 4. Imports

In [ ]:
import asyncio
import aiohttp
import json
import re
import gc
from typing import List, Optional, Dict, Any, TypedDict
from enum import Enum
from threading import Thread
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from tavily import AsyncTavilyClient
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)

print("All imports loaded.")

## 5. Load MedGemma (4-bit Quantized)

In [ ]:
MODEL_ID = "google/medgemma-4b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(MODEL_ID)

try:
    qconfig = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=qconfig, device_map="auto"
    )
    print("Loaded with 4-bit quantization")
except Exception as e:
    print(f"Quantization failed ({e}), loading with bfloat16")
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    )

model.eval()

if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 6. MedGemma Inference Function

In [ ]:
def query_medgemma(prompt: str, max_tokens: int = 1024, stream: bool = True, callback=None) -> str:
    """Query MedGemma using proper chat template format."""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)

    streamer = TextIteratorStreamer(
        processor.tokenizer, skip_prompt=True, skip_special_tokens=True
    )

    gen_kwargs = dict(**inputs, max_new_tokens=max_tokens, do_sample=False, streamer=streamer)
    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    full_response = ""
    for token in streamer:
        full_response += token
        if stream:
            print(token, end="", flush=True)
        if callback:
            callback(token)

    thread.join()
    if stream:
        print()
    return full_response.strip()

# Quick test
print("Testing MedGemma...")
test = query_medgemma("What is hypertension?", max_tokens=50, stream=True)
print(f"\nResponse length: {len(test)} chars")

## 7. Data Models

In [ ]:
class ExperienceLevel(str, Enum):
    STUDENT = "student"
    RESIDENT = "resident"
    ATTENDING = "attending"
    SENIOR = "senior"


class DoctorContext(BaseModel):
    id: str
    name: str
    specialty: str
    experience_level: ExperienceLevel
    country: str
    workplace_type: str
    workplace_name: str
    language: str = "English"
    preferences: Dict[str, Any] = Field(default_factory=dict)


class LabResult(BaseModel):
    value: Any
    unit: str = ""
    reference: str = ""
    flag: str = "N"


class Medication(BaseModel):
    name: str
    dose: str = ""
    frequency: str = ""
    indication: str = ""


class MedicalCondition(BaseModel):
    condition: str
    diagnosed_year: int = 0
    status: str = "active"


class ImagingResult(BaseModel):
    type: str
    date: str = ""
    findings: str = ""


class PatientEHR(BaseModel):
    patient_id: str
    name: str
    age: int
    sex: str
    weight_kg: float = 70.0
    height_cm: float = 170.0
    bmi: float = 24.2
    country: str = "USA"
    insurance: str = "Unknown"
    allergies: List[str] = Field(default_factory=list)
    current_medications: List[Medication] = Field(default_factory=list)
    medical_history: List[MedicalCondition] = Field(default_factory=list)
    surgical_history: List[str] = Field(default_factory=list)
    family_history: List[str] = Field(default_factory=list)
    social_history: Dict[str, Any] = Field(default_factory=dict)
    recent_labs: Dict[str, LabResult] = Field(default_factory=dict)
    recent_vitals: Dict[str, str] = Field(default_factory=dict)
    recent_imaging: List[ImagingResult] = Field(default_factory=list)
    chief_complaint: str = ""
    hpi: str = ""
    ros: Dict[str, List[str]] = Field(default_factory=dict)


print("Data models defined.")

## 8. Doctor Personas (USA + Germany)

In [ ]:
DOCTOR_PERSONAS = {
    "usa_er_attending": DoctorContext(
        id="usa_er_attending", name="Dr. Sarah Chen",
        specialty="Emergency Medicine", experience_level=ExperienceLevel.ATTENDING,
        country="USA", workplace_type="Urban Academic Hospital",
        workplace_name="Massachusetts General Hospital",
        preferences={"response_style": "concise", "wants_disposition": True},
    ),
    "usa_fm_resident": DoctorContext(
        id="usa_fm_resident", name="Dr. Michael Torres",
        specialty="Family Medicine", experience_level=ExperienceLevel.RESIDENT,
        country="USA", workplace_type="Rural Community Clinic",
        workplace_name="Riverside Family Health Center",
        preferences={"response_style": "educational", "limited_resources": True},
    ),
    "usa_med_student": DoctorContext(
        id="usa_med_student", name="James Wilson (MS3)",
        specialty="Internal Medicine (Rotation)", experience_level=ExperienceLevel.STUDENT,
        country="USA", workplace_type="Teaching Hospital",
        workplace_name="Johns Hopkins Hospital",
        preferences={"response_style": "detailed_educational"},
    ),
    "de_internist": DoctorContext(
        id="de_internist", name="Dr. Hans Muller",
        specialty="Innere Medizin", experience_level=ExperienceLevel.ATTENDING,
        country="Germany", workplace_type="University Hospital",
        workplace_name="Charite - Universitatsmedizin Berlin",
        language="German",
        preferences={"response_style": "concise", "uses_european_guidelines": True},
    ),
    "de_hausarzt": DoctorContext(
        id="de_hausarzt", name="Dr. Anna Schmidt",
        specialty="Allgemeinmedizin", experience_level=ExperienceLevel.SENIOR,
        country="Germany", workplace_type="General Practice",
        workplace_name="Praxis Dr. Schmidt, Munchen",
        language="German",
        preferences={"response_style": "practical", "outpatient_focus": True},
    ),
    "de_pj_student": DoctorContext(
        id="de_pj_student", name="Lisa Weber (PJ)",
        specialty="Innere Medizin (PJ Tertial)", experience_level=ExperienceLevel.STUDENT,
        country="Germany", workplace_type="Teaching Hospital",
        workplace_name="Universitatsklinikum Heidelberg",
        language="German",
        preferences={"response_style": "detailed_educational"},
    ),
}

print(f"Doctor personas: {list(DOCTOR_PERSONAS.keys())}")

## 9. Patient EHR Data (USA + Germany)

In [ ]:
USA_PATIENTS = {
    "USA_P001": PatientEHR(
        patient_id="USA_P001", name="Robert Johnson", age=67, sex="Male",
        weight_kg=92.5, height_cm=178, bmi=29.2, country="USA", insurance="Medicare",
        allergies=["Penicillin (rash)", "Sulfa (anaphylaxis)", "Shellfish"],
        current_medications=[
            Medication(name="Metformin", dose="1000mg", frequency="BID", indication="T2DM"),
            Medication(name="Lisinopril", dose="20mg", frequency="daily", indication="HTN"),
            Medication(name="Atorvastatin", dose="40mg", frequency="daily", indication="HLD"),
            Medication(name="Aspirin", dose="81mg", frequency="daily", indication="CAD prevention"),
            Medication(name="Metoprolol succinate", dose="50mg", frequency="daily", indication="HTN/CAD"),
        ],
        medical_history=[
            MedicalCondition(condition="Type 2 Diabetes Mellitus", diagnosed_year=2015, status="controlled"),
            MedicalCondition(condition="Hypertension", diagnosed_year=2010, status="controlled"),
            MedicalCondition(condition="Hyperlipidemia", diagnosed_year=2012, status="on treatment"),
            MedicalCondition(condition="Coronary Artery Disease", diagnosed_year=2020, status="s/p PCI to LAD"),
            MedicalCondition(condition="CKD Stage 3a", diagnosed_year=2022, status="stable"),
        ],
        surgical_history=["PCI with DES to LAD (2020)", "Appendectomy (1985)"],
        family_history=["Father: MI at 58", "Mother: T2DM, HTN", "Brother: T2DM"],
        social_history={"smoking": "Former, quit 2020, 30 pack-years", "alcohol": "Occasional", "occupation": "Retired accountant"},
        recent_labs={
            "HbA1c": LabResult(value=7.8, unit="%", reference="<7.0", flag="H"),
            "Creatinine": LabResult(value=1.4, unit="mg/dL", reference="0.7-1.3", flag="H"),
            "eGFR": LabResult(value=52, unit="mL/min", reference=">60", flag="L"),
            "Potassium": LabResult(value=5.1, unit="mEq/L", reference="3.5-5.0", flag="H"),
            "NT-proBNP": LabResult(value=450, unit="pg/mL", reference="<300", flag="H"),
            "Hemoglobin": LabResult(value=12.8, unit="g/dL", reference="13.5-17.5", flag="L"),
            "LDL": LabResult(value=95, unit="mg/dL", reference="<70 (CAD)", flag="H"),
        },
        recent_vitals={"BP": "148/92 mmHg", "HR": "78 bpm", "RR": "16/min", "SpO2": "96% on RA", "Temp": "98.4F"},
        recent_imaging=[
            ImagingResult(type="Echo", date="2025-11-15", findings="EF 45%, mild LVH, grade 1 diastolic dysfunction"),
            ImagingResult(type="CXR", date="2025-12-20", findings="Mild cardiomegaly, no acute infiltrates"),
        ],
        chief_complaint="Increasing shortness of breath and leg swelling for 2 weeks",
        hpi="67M with T2DM, HTN, CAD s/p PCI presents with 2 weeks of progressive dyspnea on exertion and bilateral LE edema. Previously walked 4 blocks, now SOB after 1 block. New 2-pillow orthopnea and occasional PND. Gained 7 lbs in 2 weeks. Ran out of Metoprolol for 5 days last week.",
        ros={"constitutional": ["fatigue", "weight gain"], "cardiovascular": ["dyspnea on exertion", "orthopnea", "PND", "leg swelling"]},
    ),
    "USA_P002": PatientEHR(
        patient_id="USA_P002", name="Maria Santos", age=34, sex="Female",
        weight_kg=68.0, height_cm=163, bmi=25.6, country="USA", insurance="Blue Cross PPO",
        allergies=["Ibuprofen (GI upset)", "Latex"],
        current_medications=[
            Medication(name="Levothyroxine", dose="75mcg", frequency="daily", indication="Hypothyroidism"),
            Medication(name="Sertraline", dose="100mg", frequency="daily", indication="Anxiety"),
            Medication(name="Vitamin D", dose="2000 IU", frequency="daily", indication="Deficiency"),
        ],
        medical_history=[
            MedicalCondition(condition="Hashimoto's Thyroiditis", diagnosed_year=2018, status="on replacement"),
            MedicalCondition(condition="Generalized Anxiety Disorder", diagnosed_year=2019, status="controlled"),
            MedicalCondition(condition="Migraine without aura", diagnosed_year=2015, status="intermittent"),
        ],
        surgical_history=["Laparoscopic cholecystectomy (2021)"],
        family_history=["Mother: Hypothyroidism, Breast cancer at 52", "Sister: SLE"],
        social_history={"smoking": "Never", "alcohol": "Social", "occupation": "Software engineer"},
        recent_labs={
            "TSH": LabResult(value=2.4, unit="mIU/L", reference="0.4-4.0", flag="N"),
            "ANA": LabResult(value="1:80", unit="titer", reference="<1:40", flag="H"),
            "ESR": LabResult(value=28, unit="mm/hr", reference="0-20", flag="H"),
            "CRP": LabResult(value=1.2, unit="mg/dL", reference="<0.5", flag="H"),
            "Hemoglobin": LabResult(value=12.1, unit="g/dL", reference="12.0-16.0", flag="N"),
        },
        recent_vitals={"BP": "118/72 mmHg", "HR": "76 bpm", "Temp": "99.1F", "SpO2": "99%"},
        chief_complaint="Joint pain and fatigue for 6 weeks",
        hpi="34F with Hashimoto's presents with 6 weeks of joint pain (hands, wrists, knees) and fatigue. Morning stiffness ~1 hour. Low-grade fevers. New photosensitivity and faint malar rash. Sister recently diagnosed with lupus.",
        ros={"constitutional": ["fatigue", "low-grade fevers"], "skin": ["photosensitivity", "malar rash"], "msk": ["polyarthralgias", "morning stiffness"]},
    ),
}

GERMANY_PATIENTS = {
    "DE_P001": PatientEHR(
        patient_id="DE_P001", name="Klaus Becker", age=72, sex="Male",
        weight_kg=85.0, height_cm=175, bmi=27.8, country="Germany", insurance="AOK Bayern",
        allergies=["Amoxicillin (Exanthem)", "Kontrastmittel (Anaphylaxie)"],
        current_medications=[
            Medication(name="Ramipril", dose="5mg", frequency="taglich", indication="Hypertonie"),
            Medication(name="Bisoprolol", dose="5mg", frequency="taglich", indication="KHK/VHF"),
            Medication(name="ASS", dose="100mg", frequency="taglich", indication="KHK"),
            Medication(name="Simvastatin", dose="40mg", frequency="abends", indication="Hyperlipidamie"),
            Medication(name="Pantoprazol", dose="20mg", frequency="taglich", indication="Reflux"),
        ],
        medical_history=[
            MedicalCondition(condition="KHK (Koronare Herzkrankheit)", diagnosed_year=2018, status="Z.n. PTCA LAD"),
            MedicalCondition(condition="Arterielle Hypertonie", diagnosed_year=2008, status="eingestellt"),
            MedicalCondition(condition="Vorhofflimmern paroxysmal", diagnosed_year=2022, status="Frequenzkontrolle"),
            MedicalCondition(condition="COPD GOLD II", diagnosed_year=2019, status="stabil"),
        ],
        surgical_history=["PTCA mit DES LAD (2018)", "Appendektomie (1965)"],
        family_history=["Vater: Herzinfarkt mit 65", "Mutter: Schlaganfall"],
        social_history={"smoking": "Ex-Raucher seit 2018, 40 Packungsjahre", "alcohol": "Gelegentlich Bier"},
        recent_labs={
            "Kreatinin": LabResult(value=1.3, unit="mg/dL", reference="0.7-1.2", flag="H"),
            "eGFR": LabResult(value=55, unit="mL/min", reference=">60", flag="L"),
            "NT-proBNP": LabResult(value=890, unit="pg/mL", reference="<300", flag="H"),
            "Kalium": LabResult(value=4.8, unit="mmol/L", reference="3.5-5.0", flag="N"),
            "Hamoglobin": LabResult(value=13.2, unit="g/dL", reference="13.5-17.5", flag="L"),
        },
        recent_vitals={"BP": "152/88 mmHg", "HR": "88/min, unregelmassig", "SpO2": "94%", "Temp": "36.8C"},
        recent_imaging=[ImagingResult(type="Echo", date="2025-10-10", findings="EF 40%, LA dilatiert, diastolische Dysfunktion Grad II")],
        chief_complaint="Zunehmende Belastungsdyspnoe und Beinodeme seit 10 Tagen",
        hpi="72-jahriger Patient mit KHK, Hypertonie, paroxysmalem VHF. Seit 10 Tagen zunehmende Belastungsdyspnoe (fruher 500m, jetzt 100m). Unterschenkelodeme beidseits, 2-Kissen-Orthopnoe (neu).",
        ros={"konstitutionell": ["Mudigkeit", "Gewichtszunahme 3kg"], "kardiovaskular": ["Belastungsdyspnoe", "Orthopnoe", "Beinodeme"]},
    ),
    "DE_P002": PatientEHR(
        patient_id="DE_P002", name="Sabine Hoffmann", age=45, sex="Female",
        weight_kg=72.0, height_cm=168, bmi=25.5, country="Germany", insurance="TK",
        allergies=["Metamizol", "Nickel"],
        current_medications=[
            Medication(name="L-Thyroxin", dose="100ug", frequency="taglich nuchtern", indication="Hypothyreose"),
            Medication(name="Citalopram", dose="20mg", frequency="taglich", indication="Depression"),
        ],
        medical_history=[
            MedicalCondition(condition="Hashimoto-Thyreoiditis", diagnosed_year=2016, status="substituiert"),
            MedicalCondition(condition="Rezidivierende depressive Storung", diagnosed_year=2018, status="unter Therapie"),
            MedicalCondition(condition="Migrane ohne Aura", diagnosed_year=2008, status="intermittierend"),
        ],
        surgical_history=["Sectio caesarea (2012)"],
        family_history=["Mutter: Hashimoto, RA", "Schwester: MS"],
        social_history={"smoking": "Nie", "alcohol": "Selten", "occupation": "Lehrerin"},
        recent_labs={
            "TSH": LabResult(value=1.8, unit="mU/L", reference="0.4-4.0", flag="N"),
            "Hamoglobin": LabResult(value=11.8, unit="g/dL", reference="12.0-16.0", flag="L"),
            "Ferritin": LabResult(value=18, unit="ng/mL", reference="15-150", flag="N"),
            "Vitamin D": LabResult(value=22, unit="ng/mL", reference="30-100", flag="L"),
        },
        recent_vitals={"BP": "125/78 mmHg", "HR": "72/min", "SpO2": "98%", "Temp": "36.6C"},
        chief_complaint="Zunehmende Mudigkeit und Kopfschmerzen seit 4 Wochen",
        hpi="45-jahrige Patientin mit Hashimoto und Depression. Seit 4 Wochen Mudigkeit und dumpfe Kopfschmerzen. Konzentrationsstorungen, Schwindel beim Aufstehen.",
        ros={"konstitutionell": ["Mudigkeit"], "neurologisch": ["Kopfschmerzen", "Schwindel"]},
    ),
}

ALL_PATIENTS = {**USA_PATIENTS, **GERMANY_PATIENTS}
print(f"Patients loaded: {list(ALL_PATIENTS.keys())}")

## 10. Async Tools (PubMed, OpenFDA, Tavily Guidelines, Allergy Check)

In [ ]:
# Geography-aware guideline domains
GUIDELINE_DOMAINS = {
    "USA": ["acc.org", "heart.org", "diabetes.org", "nih.gov", "cdc.gov", "uptodate.com", "fda.gov"],
    "Germany": ["awmf.org", "escardio.org", "dgk.org", "aerzteblatt.de", "ema.europa.eu"],
    "EU": ["escardio.org", "ema.europa.eu", "nice.org.uk", "easl.eu"],
    "India": ["icmr.nic.in", "cdsco.gov.in", "apiindia.org"],
    "UK": ["nice.org.uk", "bnf.nice.org.uk", "gov.uk"],
}

# Cross-reactivity map
CROSS_REACTIVITY = {
    "penicillin": ["amoxicillin", "ampicillin", "piperacillin", "nafcillin"],
    "sulfa": ["sulfamethoxazole", "sulfasalazine", "trimethoprim-sulfamethoxazole"],
    "cephalosporin": ["cephalexin", "ceftriaxone", "cefazolin", "cefepime"],
    "nsaid": ["ibuprofen", "naproxen", "ketorolac", "diclofenac", "celecoxib"],
}


async def search_pubmed(terms: List[str], max_results: int = 5) -> List[Dict]:
    """Search PubMed for medical literature."""
    results = []
    query = "+".join(terms[:5])
    timeout = aiohttp.ClientTimeout(total=15)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        search_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term={query}&retmax={max_results}&retmode=json"
        async with session.get(search_url) as resp:
            if resp.status != 200:
                return results
            data = await resp.json()
        ids = data.get("esearchresult", {}).get("idlist", [])
        if not ids:
            return results
        summ_url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id={','.join(ids)}&retmode=json"
        async with session.get(summ_url) as resp:
            if resp.status != 200:
                return results
            summaries = await resp.json()
        for pmid in ids:
            article = summaries.get("result", {}).get(pmid, {})
            if isinstance(article, dict):
                results.append({
                    "title": article.get("title", ""),
                    "authors": ", ".join(a.get("name", "") for a in article.get("authors", [])[:3]),
                    "source": article.get("source", ""),
                    "pubdate": article.get("pubdate", ""),
                    "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                })
    return results


async def search_openfda(drug_names: List[str]) -> List[Dict]:
    """Search OpenFDA for drug safety information."""
    timeout = aiohttp.ClientTimeout(total=10)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async def fetch(drug):
            name = drug.split()[0].lower()
            url = f"https://api.fda.gov/drug/label.json?search=openfda.generic_name:{name}&limit=1"
            try:
                async with session.get(url) as resp:
                    if resp.status != 200:
                        return None
                    data = await resp.json()
                    r = data.get("results", [None])[0]
                    if not r:
                        return None
                    return {
                        "drug": name,
                        "warnings": (r.get("warnings") or [""])[:1],
                        "contraindications": (r.get("contraindications") or [""])[:1],
                        "drug_interactions": (r.get("drug_interactions") or [""])[:1],
                    }
            except Exception:
                return None
        fetched = await asyncio.gather(*[fetch(d) for d in drug_names[:5]])
    return [r for r in fetched if r]


async def search_guidelines(query: str, country: str = "USA") -> List[Dict]:
    """Search for clinical guidelines using Tavily (geography-aware)."""
    if not TAVILY_API_KEY:
        return []
    try:
        client = AsyncTavilyClient(api_key=TAVILY_API_KEY)
        domains = GUIDELINE_DOMAINS.get(country, GUIDELINE_DOMAINS["USA"])
        response = await client.search(
            query=f"clinical guidelines {query}",
            search_depth="advanced",
            include_domains=domains,
            max_results=5,
        )
        return [{
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": r.get("content", "")[:400],
        } for r in response.get("results", [])]
    except Exception:
        return []


async def check_allergies(medications: List[str], allergies: List[str]) -> List[Dict]:
    """Check for allergy conflicts including cross-reactivity."""
    alerts = []
    allergy_keywords = [a.split()[0].lower() for a in allergies]
    for med in medications:
        med_lower = med.lower()
        for idx, keyword in enumerate(allergy_keywords):
            if keyword in med_lower:
                alerts.append({"drug": med, "allergy": allergies[idx], "severity": "high"})
                continue
            for family, members in CROSS_REACTIVITY.items():
                if keyword in family or family in keyword:
                    if any(m in med_lower for m in members):
                        alerts.append({"drug": med, "allergy": f"{allergies[idx]} (cross-reactivity: {family})", "severity": "moderate"})
    return alerts


print("Async tools defined.")

## 11. Prompt Templates

In [ ]:
STYLE_MAP = {
    "student": "Provide detailed explanations with pathophysiology, step-by-step reasoning, and learning points.",
    "resident": "Focus on clinical decision-making, key decision points, evidence-based reasoning.",
    "attending": "Be concise and action-oriented. Focus on critical findings and next steps.",
    "senior": "Brief summary. Critical points only. New evidence that changes practice.",
}

CRITICAL_MODE_PREFIX = """CRITICAL CARE MODE

STRICT FORMAT - EXACTLY 4 LINES:
Line 1: Assessment (diagnosis/condition with key finding)
Line 2: Immediate action (drug + exact dose OR intervention)
Line 3: Monitor (specific parameter + target value)
Line 4: Context-aware question (what you need to know for next step)

"""

INTENT_PROMPT = """Analyse this medical query. Respond with ONLY JSON.

Query: "{query}"
Has patient: {has_patient}

{{"needs_drugs": true/false, "needs_literature": true/false, "needs_guidelines": true/false, "search_terms": ["term1", "term2"]}}

JSON:"""


def build_reasoner_prompt(query, doctor, patient=None, allergy_alerts=None,
                          guideline_results=None, pubmed_results=None,
                          openfda_results=None, mode="regular"):
    """Build the full prompt with all context and evidence."""
    experience = doctor.experience_level.value
    style = STYLE_MAP.get(experience, STYLE_MAP["attending"])
    parts = []

    if mode == "critical":
        parts.append(CRITICAL_MODE_PREFIX)

    parts.append(
        f"You are ContextMed, a clinical AI assistant.\n"
        f"Tailor to {doctor.specialty} ({experience}) in {doctor.country}.\n"
        f"Respond in {doctor.language}.\n{style}"
    )

    parts.append(f"\nPHYSICIAN: {doctor.name} | {doctor.specialty} | {experience} | {doctor.country}")

    if patient:
        meds = ", ".join(f"{m.name} {m.dose}" for m in patient.current_medications[:5])
        conditions = ", ".join(h.condition for h in patient.medical_history[:5])
        labs_abn = [
            f"{k}: {v.value}{v.unit} ({v.flag})"
            for k, v in patient.recent_labs.items() if v.flag != "N"
        ]
        vitals = ", ".join(f"{k}: {v}" for k, v in patient.recent_vitals.items())
        parts.append(
            f"\nPATIENT: {patient.age}yo {patient.sex}, BMI {patient.bmi}\n"
            f"CC: {patient.chief_complaint}\n"
            f"Hx: {conditions}\nMeds: {meds}\n"
            f"ALLERGIES: {', '.join(patient.allergies) or 'NKDA'}\n"
            f"Vitals: {vitals}\nAbnormal Labs: {', '.join(labs_abn[:5]) or 'None'}\n"
            f"HPI: {patient.hpi}"
        )

    if allergy_alerts:
        parts.append("\nALLERGY ALERTS:")
        for a in allergy_alerts:
            parts.append(f"  - {a['drug']} conflicts with {a['allergy']}")

    if guideline_results:
        parts.append("\nGUIDELINES:")
        for r in guideline_results[:3]:
            parts.append(f"  - {r['title'][:80]}\n    {r.get('content', '')[:200]}")

    if pubmed_results:
        parts.append("\nLITERATURE:")
        for r in pubmed_results[:3]:
            parts.append(f"  - {r['title'][:80]} ({r.get('source', '')}, {r.get('pubdate', '')})")

    if openfda_results:
        parts.append("\nDRUG SAFETY:")
        for r in openfda_results[:3]:
            parts.append(f"  - {r['drug'].title()}: See warnings/interactions")

    parts.append(
        f"\n---\nQUESTION: {query}\n---\n\n"
        f"Respond with: 1) ASSESSMENT 2) DIFFERENTIAL 3) RECOMMENDATIONS "
        f"4) SAFETY 5) FOLLOW-UP\n\nResponse:"
    )

    return "\n".join(parts)


print("Prompt templates defined.")

## 12. Conversation Memory

In [ ]:
from collections import defaultdict

class ConversationMemory:
    """Per-session conversation memory for multi-turn reasoning."""

    def __init__(self, max_turns=20):
        self._store = defaultdict(list)
        self.max_turns = max_turns

    def _key(self, doctor_id, patient_id=""):
        return f"{doctor_id}::{patient_id}"

    def add(self, doctor_id, role, content, patient_id=""):
        key = self._key(doctor_id, patient_id)
        self._store[key].append({"role": role, "content": content})
        if len(self._store[key]) > self.max_turns * 2:
            self._store[key] = self._store[key][-self.max_turns * 2:]

    def get_history(self, doctor_id, patient_id="", last_n=6):
        return self._store[self._key(doctor_id, patient_id)][-last_n:]

    def format_for_prompt(self, doctor_id, patient_id="", last_n=6):
        history = self.get_history(doctor_id, patient_id, last_n)
        if not history:
            return ""
        lines = ["PREVIOUS CONVERSATION:"]
        for msg in history:
            role = "Doctor" if msg["role"] == "user" else "Assistant"
            lines.append(f"{role}: {msg['content'][:300]}")
        return "\n".join(lines)

    def clear(self, doctor_id, patient_id=""):
        self._store[self._key(doctor_id, patient_id)] = []


memory = ConversationMemory()
print("Conversation memory initialized.")

## 13. LangGraph Agentic Workflow

Pipeline: **Planner** -> **Retriever** -> **Reasoner** -> **Formatter**

- Planner uses MedGemma to classify intent and select tools
- Retriever runs all tools in parallel (async)
- Reasoner generates evidence-grounded clinical response
- Formatter adds safety alerts, citations, and metadata

In [ ]:
class AgentState(TypedDict):
    query: str
    doctor: Dict[str, Any]
    patient: Optional[Dict[str, Any]]
    mode: str
    conversation_context: str
    needs_drugs: bool
    needs_literature: bool
    needs_guidelines: bool
    search_terms: List[str]
    pubmed_results: List[Dict]
    openfda_results: List[Dict]
    guideline_results: List[Dict]
    allergy_alerts: List[Dict]
    final_response: str
    citations: List[Dict]
    tools_used: List[str]


def planner_node(state: AgentState) -> AgentState:
    """Analyse query and decide which tools to run using MedGemma."""
    print("PLANNER: Analysing query...")
    query = state["query"]
    has_patient = state.get("patient") is not None

    prompt = INTENT_PROMPT.format(query=query, has_patient="Yes" if has_patient else "No")
    try:
        resp = query_medgemma(prompt, max_tokens=120, stream=False)
        match = re.search(r"\{[^{}]+\}", resp)
        if match:
            intent = json.loads(match.group())
            state["needs_drugs"] = intent.get("needs_drugs", False) or has_patient
            state["needs_literature"] = intent.get("needs_literature", True)
            state["needs_guidelines"] = intent.get("needs_guidelines", True)
            terms = intent.get("search_terms", [])
            state["search_terms"] = terms if terms else query.split()[:5]
        else:
            raise ValueError("No JSON")
    except Exception:
        state["needs_drugs"] = has_patient
        state["needs_literature"] = True
        state["needs_guidelines"] = True
        state["search_terms"] = query.split()[:5]

    tools = []
    if state["needs_drugs"]: tools.append("OpenFDA")
    if state["needs_literature"]: tools.append("PubMed")
    if state["needs_guidelines"]: tools.append("Guidelines")
    if has_patient: tools.append("AllergyCheck")
    print(f"   Tools planned: {tools} | Terms: {state['search_terms'][:3]}")
    return state


async def _retrieve_async(state):
    """Run all tools in parallel."""
    tasks, names = [], []
    patient = state.get("patient")
    doctor = state.get("doctor", {})
    country = doctor.get("country", "USA")
    terms = state.get("search_terms", [])

    if state.get("needs_literature") and terms:
        tasks.append(search_pubmed(terms))
        names.append("pubmed")
    if state.get("needs_drugs") and patient:
        meds = [m.get("name", "") for m in patient.get("current_medications", [])]
        if meds:
            tasks.append(search_openfda(meds))
            names.append("openfda")
    if state.get("needs_guidelines") and terms:
        tasks.append(search_guidelines(" ".join(terms[:3]), country))
        names.append("guidelines")
    if patient:
        meds = [m.get("name", "") for m in patient.get("current_medications", [])]
        allergies = patient.get("allergies", [])
        if meds and allergies:
            tasks.append(check_allergies(meds, allergies))
            names.append("allergy")

    tools_used = []
    if tasks:
        results = await asyncio.gather(*tasks, return_exceptions=True)
        for i, name in enumerate(names):
            result = results[i] if not isinstance(results[i], Exception) else []
            if name == "pubmed":
                state["pubmed_results"] = result
                if result: tools_used.append("PubMed")
            elif name == "openfda":
                state["openfda_results"] = result
                if result: tools_used.append("OpenFDA")
            elif name == "guidelines":
                state["guideline_results"] = result
                if result: tools_used.append("Guidelines")
            elif name == "allergy":
                state["allergy_alerts"] = result
                if result: tools_used.append("AllergyCheck")
    state["tools_used"] = tools_used
    return state


def retriever_node(state):
    """Sync wrapper for async retriever."""
    print("RETRIEVER: Running tools in parallel...")
    loop = asyncio.new_event_loop()
    try:
        state = loop.run_until_complete(_retrieve_async(state))
    finally:
        loop.close()
    print(f"   Tools completed: {state['tools_used']}")
    return state


def reasoner_node(state):
    """Generate clinical response grounded in retrieved evidence."""
    print("REASONER: Generating response...")
    doctor = DoctorContext(**state["doctor"])
    patient = PatientEHR(**state["patient"]) if state.get("patient") else None
    mode = state.get("mode", "regular")

    prompt = build_reasoner_prompt(
        query=state["query"], doctor=doctor, patient=patient,
        allergy_alerts=state.get("allergy_alerts"),
        guideline_results=state.get("guideline_results"),
        pubmed_results=state.get("pubmed_results"),
        openfda_results=state.get("openfda_results"),
        mode=mode,
    )

    conv = state.get("conversation_context", "")
    if conv:
        prompt = conv + "\n\n" + prompt

    max_tokens = 150 if mode == "critical" else 1024
    state["final_response"] = query_medgemma(prompt, max_tokens=max_tokens, stream=True)
    return state


def formatter_node(state):
    """Add safety alerts, citations, and metadata."""
    print("FORMATTER: Finalising output...")
    response = state.get("final_response", "")

    if state.get("allergy_alerts"):
        warnings = "\n".join(f"- **{a['drug']}** conflicts with {a['allergy']}" for a in state["allergy_alerts"])
        response = f"## ALLERGY ALERTS\n{warnings}\n\n---\n\n{response}"

    citations = []
    for r in state.get("guideline_results", [])[:3]:
        citations.append({"title": r["title"][:60], "url": r["url"], "source": "guidelines"})
    for r in state.get("pubmed_results", [])[:3]:
        citations.append({"title": r["title"][:60], "url": r["url"], "source": "pubmed"})

    if citations:
        response += "\n\n---\n## References\n"
        for i, c in enumerate(citations, 1):
            response += f"{i}. [{c['title']}]({c['url']})\n"

    if state.get("tools_used"):
        response += f"\n*Tools used: {', '.join(state['tools_used'])}*"

    state["final_response"] = response
    state["citations"] = citations
    return state


# Build the graph
def build_graph():
    workflow = StateGraph(AgentState)
    workflow.add_node("planner", planner_node)
    workflow.add_node("retriever", retriever_node)
    workflow.add_node("reasoner", reasoner_node)
    workflow.add_node("formatter", formatter_node)
    workflow.set_entry_point("planner")
    workflow.add_edge("planner", "retriever")
    workflow.add_edge("retriever", "reasoner")
    workflow.add_edge("reasoner", "formatter")
    workflow.add_edge("formatter", END)
    return workflow.compile()

agent_graph = build_graph()
print("LangGraph workflow built: Planner -> Retriever -> Reasoner -> Formatter")

## 14. ContextMed Agent Class

In [ ]:
class ContextMedAgent:
    """Agentic clinical decision-support agent."""

    def __init__(self):
        self.graph = agent_graph
        self.doctors = dict(DOCTOR_PERSONAS)
        self.patients = dict(ALL_PATIENTS)
        self.doctor = None
        self.patient = None

    def set_doctor(self, doctor_id):
        if doctor_id in self.doctors:
            self.doctor = self.doctors[doctor_id]
            return f"Doctor: {self.doctor.name} ({self.doctor.specialty}, {self.doctor.country})"
        return f"Not found. Options: {list(self.doctors.keys())}"

    def set_patient(self, patient_id):
        if patient_id in self.patients:
            self.patient = self.patients[patient_id]
            return f"Patient: {self.patient.name} - {self.patient.chief_complaint}"
        return f"Not found. Options: {list(self.patients.keys())}"

    def query(self, query, mode="regular"):
        if not self.doctor:
            return "Set a doctor first: agent.set_doctor('usa_er_attending')"

        patient_id = self.patient.patient_id if self.patient else ""
        conv_context = memory.format_for_prompt(self.doctor.id, patient_id)

        print(f"\n{'='*60}")
        print(f"Doctor: {self.doctor.name} ({self.doctor.experience_level.value}, {self.doctor.country})")
        if self.patient:
            print(f"Patient: {self.patient.name} ({self.patient.age}yo {self.patient.sex})")
        print(f"Query: {query}")
        print(f"{'='*60}\n")

        state = {
            "query": query,
            "doctor": self.doctor.model_dump(),
            "patient": self.patient.model_dump() if self.patient else None,
            "mode": mode,
            "conversation_context": conv_context,
            "needs_drugs": False, "needs_literature": False, "needs_guidelines": False,
            "search_terms": [], "pubmed_results": [], "openfda_results": [],
            "guideline_results": [], "allergy_alerts": [],
            "final_response": "", "citations": [], "tools_used": [],
        }

        result = self.graph.invoke(state)

        memory.add(self.doctor.id, "user", query, patient_id)
        memory.add(self.doctor.id, "assistant", result["final_response"][:500], patient_id)

        return result["final_response"]

    def create_doctor(self, name, specialty, experience_level, country="USA",
                      workplace_type="Hospital", workplace_name="", language="English"):
        exp_map = {"student": ExperienceLevel.STUDENT, "resident": ExperienceLevel.RESIDENT,
                   "attending": ExperienceLevel.ATTENDING, "senior": ExperienceLevel.SENIOR}
        doc_id = f"custom_{len(self.doctors)}"
        doc = DoctorContext(
            id=doc_id, name=name, specialty=specialty,
            experience_level=exp_map.get(experience_level.lower(), ExperienceLevel.ATTENDING),
            country=country, workplace_type=workplace_type,
            workplace_name=workplace_name or specialty, language=language,
        )
        self.doctors[doc_id] = doc
        return doc

    def parse_ehr(self, ehr_text):
        """Parse unstructured EHR text into structured patient using MedGemma."""
        prompt = (
            f"Parse this clinical note into JSON with keys: name, age, sex, "
            f"chief_complaint, allergies, current_medications, medical_history, hpi.\n\n"
            f"{ehr_text[:2000]}\n\nJSON:"
        )
        resp = query_medgemma(prompt, max_tokens=800, stream=False)
        try:
            start, end = resp.find('{'), resp.rfind('}') + 1
            if start >= 0 and end > start:
                return json.loads(resp[start:end])
        except Exception:
            pass
        return {"name": "Unknown", "chief_complaint": ehr_text[:100], "hpi": ehr_text[:500]}


agent = ContextMedAgent()
print("ContextMed Agent ready!")
print(f"Doctors: {list(agent.doctors.keys())}")
print(f"Patients: {list(agent.patients.keys())}")

## 15. Demo: USA Attending — Heart Failure Case

In [ ]:
print(agent.set_doctor("usa_er_attending"))
print(agent.set_patient("USA_P001"))
response = agent.query("What is the differential diagnosis and management plan?")
print("\n" + "="*60)
print(response)

## 16. Demo: German Internist — Same-Type Case, Different Guidelines

Same clinical scenario (heart failure), but now with a German physician.
Note how guidelines shift from ACC/AHA to ESC/AWMF.

In [ ]:
print(agent.set_doctor("de_internist"))
print(agent.set_patient("DE_P001"))
response = agent.query("Was ist die Differentialdiagnose und der Behandlungsplan?")
print("\n" + "="*60)
print(response)

## 17. Demo: Medical Student — Educational Mode

In [ ]:
print(agent.set_doctor("usa_med_student"))
print(agent.set_patient("USA_P002"))
response = agent.query("This patient might have lupus. Can you explain the pathophysiology and diagnostic criteria?")
print("\n" + "="*60)
print(response)

## 18. Demo: Critical Care Mode

In [ ]:
print(agent.set_doctor("usa_er_attending"))
print(agent.set_patient("USA_P001"))
response = agent.query(
    "Patient found unresponsive, BP 80/50, HR 120. What do I do immediately?",
    mode="critical"
)
print("\n" + "="*60)
print(response)

## 19. Demo: Multi-Turn Conversation Memory

In [ ]:
print(agent.set_doctor("usa_er_attending"))
print(agent.set_patient("USA_P001"))

print("\n--- Turn 1 ---")
r1 = agent.query("Start IV furosemide. What dose given his renal function?")
print(r1)

print("\n--- Turn 2 (follow-up, uses memory) ---")
r2 = agent.query("The patient responded well. Urine output improved. What next?")
print(r2)

## 20. FastAPI Server (for frontend / live demo)

In [ ]:
import queue
import threading
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse

app = FastAPI(title="ContextMed API", version="2.0.0",
              description="Globally informed. Locally accurate. Clinical AI.")

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


class APIQueryRequest(BaseModel):
    query: str
    doctor_id: str
    patient_id: Optional[str] = None
    mode: str = "regular"
    conversation_history: List[Dict] = Field(default_factory=list)


@app.get("/")
def root():
    return {"name": "ContextMed", "tagline": "Globally informed. Locally accurate.", "version": "2.0.0"}

@app.get("/doctors")
def get_doctors():
    return [{"id": k, "name": v.name, "specialty": v.specialty,
             "experience_level": v.experience_level.value, "country": v.country}
            for k, v in agent.doctors.items()]

@app.get("/patients")
def get_patients():
    return [{"id": k, "name": v.name, "chief_complaint": v.chief_complaint,
             "age": v.age, "sex": v.sex, "country": v.country}
            for k, v in agent.patients.items()]

@app.get("/patient/{patient_id}")
def get_patient(patient_id: str):
    if patient_id in agent.patients:
        return agent.patients[patient_id].model_dump()
    return {"error": "Patient not found"}

@app.post("/ask/stream")
async def ask_stream(req: APIQueryRequest):
    if req.doctor_id not in agent.doctors:
        return {"error": f"Doctor not found: {req.doctor_id}"}

    agent.set_doctor(req.doctor_id)
    if req.patient_id:
        agent.set_patient(req.patient_id)
    else:
        agent.patient = None

    token_queue = queue.Queue()

    def run():
        try:
            token_queue.put(("status", "processing"))
            result = agent.query(req.query, mode=req.mode)
            for char in result:
                token_queue.put(("token", char))
            token_queue.put(("done", {
                "doctor": agent.doctor.name,
                "patient": agent.patient.name if agent.patient else None,
                "mode": req.mode,
            }))
        except Exception as e:
            token_queue.put(("error", str(e)))

    threading.Thread(target=run, daemon=True).start()

    async def generate():
        while True:
            try:
                msg_type, content = token_queue.get(timeout=0.1)
                if msg_type == "status":
                    yield f"data: {json.dumps({'status': content, 'done': False})}\n\n"
                elif msg_type == "token":
                    yield f"data: {json.dumps({'chunk': content, 'done': False})}\n\n"
                elif msg_type == "done":
                    yield f"data: {json.dumps({'done': True, **content})}\n\n"
                    break
                elif msg_type == "error":
                    yield f"data: {json.dumps({'error': content, 'done': True})}\n\n"
                    break
            except queue.Empty:
                yield ": keepalive\n\n"
                await asyncio.sleep(0.05)

    return StreamingResponse(generate(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "Connection": "keep-alive"})

print("FastAPI app defined.")

## 21. Launch Server with Ngrok

In [ ]:
from pyngrok import ngrok
from hypercorn.config import Config
from hypercorn.asyncio import serve
import time

# Kill any existing tunnels
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    ngrok.kill()
except Exception:
    pass

time.sleep(1)
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)

print("=" * 60)
print("ContextMed API is live!")
print("=" * 60)
print(f"Public URL: {public_url.public_url}")
print(f"API Docs:   {public_url.public_url}/docs")
print(f"Streaming:  POST {public_url.public_url}/ask/stream")
print("=" * 60)

config = Config()
config.bind = ["0.0.0.0:8000"]

loop = asyncio.get_event_loop()
loop.run_until_complete(serve(app, config))